# Split Stable Starts Input File By DDF

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-17
- **Last update:** 2026-07-17

## Import

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# 1. Récupérer le logger racine (ou créez un logger spécifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Définir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Éviter la duplication des handlers si la cellule est exécutée plusieurs fois
if not log.handlers:
    # 4. Créer un handler qui écrit vers la sortie standard (capturée par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Définir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour vérifier que ça fonctionne
log.info("Le logging est configuré et fonctionne dans le notebook !")

In [ ]:
# ── Output directories ───────────────────────────────────────────────────
NB_TAG = "MATCHSRC_01"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_in"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
log.info(f"Figs       : {os.path.abspath(DIR_FIGS)}")


# Output paths
out_lc_csv = os.path.join(DIR_DATA_OUT, "all_stars_lightcurves.csv")
out_sum_csv = os.path.join(DIR_DATA_OUT, "lightcurve_match_summary.csv")
dir_per_star = os.path.join(DIR_DATA_OUT, "per_star")

# ── Matplotlib style ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

In [ ]:
# ── Input targets ─────────────────────────────────────────────────────
# target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"
target_file_in = "master_stable_stars_V17-21_r2.0deg.csv"
target_file_rootname_out = target_file_in.replace(".csv", "")

In [ ]:
target_path = os.path.join(DIR_DATA_IN, target_file_in)
df_targets = pd.read_csv(target_path)
display(df_targets.head())
log.info("Loaded %d targets from %s", len(df_targets), target_path)

In [ ]:
log.info("Loaded %d targets from %s", len(df_targets), target_path)

In [ ]:
# Compter le nombre d'étoiles par field
counts = df_targets["field"].value_counts().sort_index()

# Barplot
plt.figure()
counts.plot(kind="bar", facecolor="b")

# Labels
plt.xlabel("Field")
plt.ylabel("Number of stars")
plt.title("Number of stars per field")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
list_of_fields = list(counts.index)

In [ ]:
for field in list_of_fields:
    target_filename_out = target_file_rootname_out + f"_{field}.csv"
    target_fullfilename_out = os.path.join(DIR_DATA_OUT, target_filename_out)

    df = df_targets[df_targets["field"] == field]
    df.to_csv(target_fullfilename_out)